# 💳 Credit Card Fraud & Sales Analytics – 2026
**Author:** Janakivarshasree  
**Dataset:** credit_card_fraud_2026.csv (20,000 transactions)  
**Objective:** Load, clean, enrich, analyse, and visualise credit card transaction data to derive business insights on sales performance and fraud patterns.

---
### Table of Contents
1. [Import Libraries](#1)
2. [Load Dataset](#2)
3. [Data Quality Check](#3)
4. [Feature Engineering – Sales Calculation](#4)
5. [Descriptive Statistics & Grouping](#5)
6. [Visualisations](#6)
7. [Business Insights & Decisions](#7)

## 1. Import Libraries <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded successfully ✅')

## 2. Load Dataset <a id='2'></a>

In [ ]:
# Load the CSV – adjust path if running from a different directory
df = pd.read_csv('../credit_card_fraud_2026.csv')

print(f'Shape : {df.shape[0]:,} rows  ×  {df.shape[1]} columns')
df.head()

In [ ]:
print('Column names:')
for c in df.columns:
    print(f'  • {c}  [{df[c].dtype}]')

## 3. Data Quality Check <a id='3'></a>

In [ ]:
# ── 3a. Missing values ──────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('✅ No missing values found – dataset is complete.')
else:
    print('⚠️  Missing values detected:')
    display(missing_df)

In [ ]:
# ── 3b. Duplicate rows ──────────────────────────────────────────────────────
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups}')
if dups > 0:
    df.drop_duplicates(inplace=True)
    print(f'  → Dropped. New shape: {df.shape}')

In [ ]:
# ── 3c. Negative / zero amounts ─────────────────────────────────────────────
bad_amounts = df[df['amount_usd'] <= 0]
print(f'Rows with amount_usd ≤ 0: {len(bad_amounts)}')
if len(bad_amounts) > 0:
    df = df[df['amount_usd'] > 0].copy()
    print('  → Removed invalid amounts.')

In [ ]:
# ── 3d. Basic statistics ─────────────────────────────────────────────────────
df.describe().round(2)

In [ ]:
# ── 3e. Categorical column summaries ────────────────────────────────────────
cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
for col in cat_cols:
    print(f'\n{col} → unique values: {df[col].nunique()}')
    print(df[col].value_counts().to_string())

## 4. Feature Engineering – Sales Calculation <a id='4'></a>

The dataset records each transaction's total `amount_usd`. To align with the **Sales = Quantity × Unit Price** model, we derive:
- `quantity` – from `txn_count_last_24h` clamped to min 1 (represents purchase volume within a shopping session).
- `unit_price` – `amount_usd / quantity` (average item price).
- `sales` – simply `amount_usd` (already the transaction total, i.e., Quantity × Unit Price).
- `day_name` – readable day-of-week label from `day_of_week` (0 = Monday).
- `time_period` – bucketed hour into Morning / Afternoon / Evening / Night.

In [ ]:
# Quantity = txn_count_last_24h, minimum 1
df['quantity'] = df['txn_count_last_24h'].clip(lower=1)

# Unit Price derived from amount / quantity
df['unit_price'] = (df['amount_usd'] / df['quantity']).round(2)

# Sales = Quantity × Unit Price  (equals amount_usd by construction)
df['sales'] = (df['quantity'] * df['unit_price']).round(2)

# Day name
day_map = {0:'Monday',1:'Tuesday',2:'Wednesday',3:'Thursday',4:'Friday',5:'Saturday',6:'Sunday'}
df['day_name'] = df['day_of_week'].map(day_map)

# Time period
def time_period(h):
    if 6 <= h < 12:  return 'Morning'
    elif 12 <= h < 17: return 'Afternoon'
    elif 17 <= h < 21: return 'Evening'
    else:             return 'Night'

df['time_period'] = df['time_of_day_hour'].apply(time_period)

# Fraud label
df['fraud_label'] = df['is_fraud'].map({1:'Fraud', 0:'Legitimate'})

print('Feature engineering complete ✅')
df[['transaction_id','amount_usd','quantity','unit_price','sales','day_name','time_period','fraud_label']].head(10)

## 5. Descriptive Statistics & Grouping <a id='5'></a>

In [ ]:
# ── 5a. Overall KPIs ─────────────────────────────────────────────────────────
total_txns   = len(df)
total_sales  = df['sales'].sum()
avg_sales    = df['sales'].mean()
fraud_txns   = df['is_fraud'].sum()
fraud_rate   = fraud_txns / total_txns * 100
total_fraud_loss = df[df['is_fraud']==1]['sales'].sum()

print('='*55)
print(f'  Total Transactions   : {total_txns:>10,}')
print(f'  Total Sales (USD)    : ${total_sales:>12,.2f}')
print(f'  Average Sales (USD)  : ${avg_sales:>12,.2f}')
print(f'  Fraud Transactions   : {fraud_txns:>10,}  ({fraud_rate:.2f}%)')
print(f'  Total Fraud Loss     : ${total_fraud_loss:>12,.2f}')
print('='*55)

In [ ]:
# ── 5b. Sales by Merchant Category ──────────────────────────────────────────
cat_summary = df.groupby('merchant_category').agg(
    Total_Transactions=('transaction_id','count'),
    Total_Sales=('sales','sum'),
    Average_Sales=('sales','mean'),
    Fraud_Count=('is_fraud','sum')
).round(2).sort_values('Total_Sales', ascending=False)
cat_summary['Fraud_Rate_%'] = (cat_summary['Fraud_Count'] / cat_summary['Total_Transactions'] * 100).round(2)
display(cat_summary)

In [ ]:
# ── 5c. Sales by Card Type ───────────────────────────────────────────────────
card_summary = df.groupby('card_type').agg(
    Total_Transactions=('transaction_id','count'),
    Total_Sales=('sales','sum'),
    Average_Sales=('sales','mean'),
    Fraud_Count=('is_fraud','sum')
).round(2).sort_values('Total_Sales', ascending=False)
display(card_summary)

In [ ]:
# ── 5d. Sales by Channel ─────────────────────────────────────────────────────
channel_summary = df.groupby('channel').agg(
    Total_Transactions=('transaction_id','count'),
    Total_Sales=('sales','sum'),
    Average_Sales=('sales','mean'),
    Fraud_Count=('is_fraud','sum')
).round(2).sort_values('Total_Sales', ascending=False)
display(channel_summary)

In [ ]:
# ── 5e. Sales by Day of Week ─────────────────────────────────────────────────
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_summary = df.groupby('day_name').agg(
    Total_Sales=('sales','sum'),
    Avg_Sales=('sales','mean'),
    Transactions=('transaction_id','count')
).reindex(day_order).round(2)
display(day_summary)

In [ ]:
# ── 5f. Fraud vs Legitimate – comparative stats ──────────────────────────────
fraud_compare = df.groupby('fraud_label').agg(
    Count=('transaction_id','count'),
    Total_Sales=('sales','sum'),
    Avg_Sales=('sales','mean'),
    Avg_Velocity=('velocity_score','mean'),
    Avg_Risk_Score=('merchant_risk_score','mean')
).round(2)
display(fraud_compare)

In [ ]:
# ── 5g. Auth method effectiveness ────────────────────────────────────────────
auth_summary = df.groupby('auth_method').agg(
    Transactions=('transaction_id','count'),
    Fraud_Count=('is_fraud','sum'),
    Total_Sales=('sales','sum')
)
auth_summary['Fraud_Rate_%'] = (auth_summary['Fraud_Count'] / auth_summary['Transactions'] * 100).round(2)
auth_summary = auth_summary.sort_values('Fraud_Rate_%')
display(auth_summary)

## 6. Visualisations <a id='6'></a>

In [ ]:
# ── 6a. Total Sales by Merchant Category (Bar Chart) ────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
data = cat_summary['Total_Sales'].sort_values(ascending=False)
colors = sns.color_palette('Blues_r', len(data))
bars = ax.bar(data.index, data.values, color=colors, edgecolor='white', linewidth=0.8)
ax.set_title('Total Sales by Merchant Category', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Merchant Category', fontsize=11)
ax.set_ylabel('Total Sales (USD)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6b. Fraud vs Legitimate – Pie Chart ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie – transaction count
counts = df['fraud_label'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#4CAF50','#F44336'], startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[0].set_title('Transaction Count:\nFraud vs Legitimate', fontweight='bold')

# Pie – sales value
sales_split = df.groupby('fraud_label')['sales'].sum()
axes[1].pie(sales_split, labels=sales_split.index, autopct='%1.1f%%',
            colors=['#4CAF50','#F44336'], startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Sales Value (USD):\nFraud vs Legitimate', fontweight='bold')

plt.suptitle('Fraud vs Legitimate Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6c. Fraud Rate by Merchant Category ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fraud_rate_cat = cat_summary['Fraud_Rate_%'].sort_values(ascending=False)
colors = ['#F44336' if v > fraud_rate else '#4CAF50' for v in fraud_rate_cat.values]
bars = ax.bar(fraud_rate_cat.index, fraud_rate_cat.values, color=colors, edgecolor='white')
ax.axhline(fraud_rate, color='navy', linestyle='--', linewidth=1.5, label=f'Overall Avg: {fraud_rate:.2f}%')
ax.set_title('Fraud Rate (%) by Merchant Category', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Fraud Rate (%)')
ax.legend()
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6d. Sales by Day of Week ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(day_order, day_summary.loc[day_order,'Total_Sales'].values,
        marker='o', linewidth=2.5, color='#3b82d4', markersize=8, markerfacecolor='white', markeredgewidth=2)
ax.fill_between(day_order, day_summary.loc[day_order,'Total_Sales'].values, alpha=0.15, color='#3b82d4')
ax.set_title('Total Sales by Day of Week', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Total Sales (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# ── 6e. Sales Distribution – Histogram ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df[df['is_fraud']==0]['sales'], bins=50, alpha=0.7, color='#4CAF50', label='Legitimate', edgecolor='white')
ax.hist(df[df['is_fraud']==1]['sales'], bins=50, alpha=0.7, color='#F44336', label='Fraud', edgecolor='white')
ax.set_title('Sales Amount Distribution: Fraud vs Legitimate', fontsize=14, fontweight='bold')
ax.set_xlabel('Transaction Amount (USD)')
ax.set_ylabel('Frequency')
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# ── 6f. Fraud Count by Auth Method ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
auth_fraud = auth_summary['Fraud_Rate_%'].sort_values(ascending=True)
colors = ['#4CAF50' if v < 5 else '#FF9800' if v < 10 else '#F44336' for v in auth_fraud.values]
ax.barh(auth_fraud.index, auth_fraud.values, color=colors, edgecolor='white')
ax.set_title('Fraud Rate (%) by Authentication Method', fontsize=13, fontweight='bold')
ax.set_xlabel('Fraud Rate (%)')
for i, v in enumerate(auth_fraud.values):
    ax.text(v + 0.1, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6g. Heatmap – Fraud by Channel × Device ──────────────────────────────────
pivot = df.pivot_table(values='is_fraud', index='channel', columns='device_type', aggfunc='mean') * 100
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot.round(1), annot=True, fmt='.1f', cmap='Reds',
            linewidths=0.5, linecolor='white', ax=ax, cbar_kws={'label':'Fraud Rate (%)'})
ax.set_title('Fraud Rate (%) – Channel × Device Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6h. Average Sales by Time Period ──────────────────────────────────────────
time_order = ['Morning','Afternoon','Evening','Night']
time_summary = df.groupby('time_period')['sales'].agg(['mean','sum','count']).reindex(time_order)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(time_order, time_summary['mean'], color=['#FFD54F','#64B5F6','#FF8A65','#7986CB'], edgecolor='white')
ax.set_title('Average Sales by Time Period', fontsize=13, fontweight='bold')
ax.set_ylabel('Average Sales (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'${bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6i. Scatter – Velocity Score vs Amount ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
legit = df[df['is_fraud']==0]
fraud_d = df[df['is_fraud']==1]
ax.scatter(legit['velocity_score'], legit['amount_usd'], alpha=0.2, s=8, color='#4CAF50', label='Legitimate')
ax.scatter(fraud_d['velocity_score'], fraud_d['amount_usd'], alpha=0.5, s=12, color='#F44336', label='Fraud')
ax.set_title('Velocity Score vs Transaction Amount', fontsize=13, fontweight='bold')
ax.set_xlabel('Velocity Score')
ax.set_ylabel('Amount (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 6j. Correlation Heatmap ───────────────────────────────────────────────────
num_cols = ['amount_usd','velocity_score','merchant_risk_score','txn_count_last_24h',
            'distance_from_home_km','account_balance_usd','customer_age','card_age_months','is_fraud']
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix – Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Business Insights & Decisions <a id='7'></a>

In [ ]:
print('='*65)
print('   BUSINESS INSIGHTS & RECOMMENDATIONS')
print('='*65)

top_cat = cat_summary['Total_Sales'].idxmax()
risky_cat = cat_summary['Fraud_Rate_%'].idxmax()
safe_auth = auth_summary['Fraud_Rate_%'].idxmin()
risky_auth = auth_summary['Fraud_Rate_%'].idxmax()

insights = [
    f"1. TOP REVENUE CATEGORY  : '{top_cat}' generates the highest total sales.",
    f"   → ACTION: Increase marketing budget and stock/service capacity for '{top_cat}'.",
    "",
    f"2. HIGHEST FRAUD RISK    : '{risky_cat}' has the highest fraud rate ({cat_summary.loc[risky_cat,'Fraud_Rate_%']:.1f}%).",
    f"   → ACTION: Apply enhanced verification (step-up auth) for all '{risky_cat}' transactions.",
    "",
    f"3. SAFEST AUTH METHOD    : '{safe_auth}' has the lowest fraud rate ({auth_summary.loc[safe_auth,'Fraud_Rate_%']:.1f}%).",
    f"   → ACTION: Promote or mandate '{safe_auth}' for high-value transactions.",
    "",
    f"4. RISKIEST AUTH METHOD  : '{risky_auth}' has fraud rate {auth_summary.loc[risky_auth,'Fraud_Rate_%']:.1f}%.",
    f"   → ACTION: Add secondary challenge or retire '{risky_auth}' for transactions > $200.",
    "",
    f"5. VELOCITY SCORE FLAG   : High velocity_score strongly correlates with fraud.",
     "   → ACTION: Auto-block or flag transactions with velocity_score > 70 for manual review.",
    "",
    f"6. PEAK SALES PERIOD     : Analyse time_period chart to schedule customer support & offers.",
     "   → ACTION: Run promotional campaigns during the highest-sales time window.",
    "",
    f"7. FOREIGN TRANSACTIONS  : Monitor is_foreign_transaction=True combined with VPN usage.",
     "   → ACTION: Require OTP/biometric for all foreign transactions with VPN detected.",
]

for line in insights:
    print(line)
print('='*65)

In [ ]:
print(f'\n📊 Final Clean Dataset Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Sales column added         : ✅')
print(f'   Visualisations created     : 10 charts')
print(f'   Analysis complete          : ✅')